
# Data Quality Pipelines and Great expectations

## Topics Covered
- What are Data Quality Pipelines?
- Why Data Quality Fails in Production
- Great Expectations (Deep Theory + Practice)
- Missing Data & Imputation Libraries
- Anomaly Detection Rules (Statistical + Business)
- End-to-End Production-Style Pipeline


## 1. What Is a Data Quality Pipeline?

A data quality pipeline is a systematic process that verifies whether incoming data satisfies predefined rules and assumptions before it is consumed by a model or analytics system.

## 1.1 Why Data Quality Pipelines Matter


In real-world systems:
- Data arrives from **multiple sources**
- Schemas evolve silently
- Business behavior changes over time

Without automated quality checks:
- Dashboards lie
- ML models silently degrade
- Business decisions fail

Instead of assuming data is correct, the pipeline explicitly checks:

- Whether required columns exist
- Whether values fall within acceptable ranges
- Whether data types are valid
- Whether distributions behave as expected

A **Data Quality Pipeline** acts as a *quality gate* between raw ingestion and downstream consumption.



## 2. Types of Data Quality Checks

### Structural Checks
- Schema validation
- Column presence
- Data types

### Content Checks
- Missing values
- Valid ranges
- Allowed categories

### Statistical Checks
- Distribution drift
- Outliers
- Sudden spikes or drops

### Business Rule Checks
- Revenue cannot be negative
- Age must be >= 18



## 3. Introduction to Great Expectations

Great Expectations is a **rule-based data validation framework**.

### Core Concepts
- Expectation: A rule that must hold true
- Expectation Suite: Collection of rules
- Validation Result: Pass/fail metrics
- Data Docs: Human-readable audit reports

Used heavily in:
- Data Engineering pipelines
- ML feature validation
- Compliance & audits

## 3.1 Where GE Fits in a Typical ML/Data Pipeline

Raw Data  →  Great Expectations  →  Feature Engg  →  ML Model  →  Decisions
                (Quality Gate)

If GE fails → stop the pipeline.
If GE passes → safe to proceed.



## 3.2. Defining Expectations (Conceptual)

Examples:
- age should not be null
- age should be between 18 and 90
- income should not be null
- country should belong to allowed set

These expectations form your **data contract**.

## 3.2. Expectation Suite

A collection of expectations for one dataset.
This represents your data contract.

Example:

Loan application dataset must satisfy these 12 rules.

## 3.3 Validation

Running data against an expectation suite to produce:

Pass / Fail

Detailed metrics (how many rows failed, where)

## 3.4) Data Docs (optional but powerful)

Human-readable HTML reports showing:

- Which rules passed
- Which failed
- Sample failing rows

Used by data engineers, ML engineers, auditors.


In [14]:
# intasll (usually done once)
!pip install great_expectations
import numpy as np
import pandas as pd
import great_expectations as gx
import warnings

warnings.filterwarnings("ignore")

print('All libraries loaded successfully✅')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 10.0 MB/s eta 0:00:00
All libraries loaded successfully✅


In [4]:
# loading the dataset I'm working on

df = pd.read_csv('dataset/cars24-car-price-cleaned-new.csv')
df.head()

,selling_price,km_driven,mileage,engine,max_power,age,make,model,Individual,Trustmark Dealer,Diesel,Electric,LPG,Petrol,Manual,5,>5
0,1.20,120000,19.70,796.0,46.30,11.0,MARUTI,ALTO STD,1,0,0,0,0,1,1,1,0
1,5.50,20000,18.90,1197.0,82.00,7.0,HYUNDAI,GRAND I10 ASTA,1,0,0,0,0,1,1,1,0
2,2.15,60000,17.00,1197.0,80.00,13.0,HYUNDAI,I20 ASTA,1,0,0,0,0,1,1,1,0
3,2.26,37000,20.92,998.0,67.10,11.0,MARUTI,ALTO K10 2010-2014 VXI,1,0,0,0,0,1,1,1,0
4,5.70,30000,22.77,1498.0,98.59,8.0,FORD,ECOSPORT 2015-2021 1.5 TDCI TITANIUM BSIV,0,0,1,0,0,0,1,1,0


In [5]:
# Converting data into great expectations object

context = gx.get_context()
data_source = context.data_sources.add_pandas('pandas')
data_asset = data_source.add_dataframe_asset(name = 'pd dataframe asset')

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmp01qq357i' for ephemeral docs site


In [6]:
from great_expectations.core import batch_definition
# build bath request (incoming data)

batch_definition = data_asset.add_batch_definition_whole_dataframe('batch definition')
batch = batch_definition.get_batch(batch_parameters={'dataframe': df})

### Create an Expectation.
Expectations are a fundamental component of GX. They allow you to explicitly define the state to which your data should conform. The following code defines an Expectation that the contents of the column `passenger_count` consist of values ranging from 2 to 6:

In [7]:
expectation = gx.expectations.ExpectColumnValuesToBeBetween(
    column='km_driven', min_value=2000, max_value=60000
)

# Run and get the results
validation_result = batch.validate(expectation)

print(validation_result)

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

{
  "success": false,
  "expectation_config": {
    "type": "expect_column_values_to_be_between",
    "kwargs": {
      "batch_id": "pandas-pd dataframe asset",
      "column": "km_driven",
      "min_value": 2000.0,
      "max_value": 60000.0
    },
    "meta": {},
    "severity": "critical"
  },
  "result": {
    "element_count": 19820,
    "unexpected_count": 7774,
    "unexpected_percent": 39.223007063572155,
    "partial_unexpected_list": [
      120000,
      70000,
      65278,
      76000,
      65000,
      62200,
      110000,
      97000,
      90000,
      77253,
      110000,
      80000,
      185000,
      90000,
      90000,
      93000,
      86000,
      120000,
      70000,
      61000
    ],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 39.223007063572155,
    "unexpected_percent_nonmissing": 39.223007063572155,
    "partial_unexpected_counts": [
      {
        "value": 90000,
        "count": 3
      },
      {
        "value"


## 4. Missing Data & Imputation Libraries

Missing data occurs due to:
- Optional fields
- Integration failures
- Late-arriving data

Types of Missing Data (Very Important Concept)

Before choosing a library, we must understand why data is missing.

- MCAR – Missing Completely At Random
  - Missingness has no pattern
  - Example: random system failure

- MAR – Missing At Random
  - Missingness depends on other columns
  - Example: income missing more for freelancers

- MNAR – Missing Not At Random
  - Missingness depends on the missing value itself
  - Example: high-income users skip income field

#### Most imputation libraries assume MCAR or MAR

### Common Strategies
- Mean / Median (numeric)
- Mode (categorical)
- Forward/Backward fill (time-series)
- Model-based imputation


In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Load dataset
df = pd.read_csv("dataset/cars24-car-price-cleaned-new.csv")

print(df.isnull().sum())

X = df.drop('selling_price', axis =1)
y = df['selling_price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.3, random_state = 42
)

selling_price       0
km_driven           0
mileage             0
engine              0
max_power           0
age                 0
make                0
model               0
Individual          0
Trustmark Dealer    0
Diesel              0
Electric            0
LPG                 0
Petrol              0
Manual              0
5                   0
>5                  0
dtype: int64


In [20]:
# Numerical - median
num_cols = X_train.select_dtypes(include = ['int64', 'float64']).columns

num_imputer_simple = SimpleImputer(strategy = 'median')

X_train_num_simple = num_imputer_simple.fit_transform(X_train[num_cols])
X_test_num_simple = num_imputer_simple.transform(X_test[num_cols])

In [21]:
# Categorical - most frequent

cat_cols = X_train.select_dtypes(include = ['object']).columns
cat_imputer = SimpleImputer(strategy = 'most_frequent')

X_train_cat = cat_imputer.fit_transform(X_train[cat_cols])
X_test_cat = cat_imputer.transform(X_test[cat_cols])

In [22]:
# Combine back
X_train_simple = pd.DataFrame(np.hstack([X_train_num_simple, X_train_cat]),
                               columns = list(num_cols) + list(cat_cols))

X_test_simple = pd.DataFrame(
    np.hstack([X_test_num_simple, X_test_cat]),
    columns = list(num_cols) + list(cat_cols)
)

X_train_simple, X_test_simple

(      km_driven mileage  engine max_power   age Individual Trustmark Dealer  \
 0       70000.0   19.01  1461.0    108.45  10.0        1.0              0.0   
 1       35000.0    16.0  2179.0     140.0   7.0        1.0              0.0   
 2       50000.0    13.9  1598.0      92.0  18.0        0.0              0.0   
 3       17000.0    18.9  1197.0      82.0   4.0        0.0              0.0   
 4       34000.0   16.38  1999.0     177.0   7.0        0.0              0.0   
 ...         ...     ...     ...       ...   ...        ...              ...   
 13869   32000.0   22.74   796.0      47.3   8.0        0.0              0.0   
 13870    3944.0   20.51   998.0      67.0   5.0        0.0              1.0   
 13871  101931.0    19.3  1248.0      73.9   9.0        0.0              0.0   
 13872   70333.0    17.8  1497.0     117.3   7.0        0.0              0.0   
 13873   90000.0   12.05  2179.0     120.0   9.0        1.0              0.0   
 
       Diesel Electric  LPG Petrol Man

In [23]:
X_test_simple.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5946 entries, 0 to 5945
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   km_driven         5946 non-null   object
 1   mileage           5946 non-null   object
 2   engine            5946 non-null   object
 3   max_power         5946 non-null   object
 4   age               5946 non-null   object
 5   Individual        5946 non-null   object
 6   Trustmark Dealer  5946 non-null   object
 7   Diesel            5946 non-null   object
 8   Electric          5946 non-null   object
 9   LPG               5946 non-null   object
 10  Petrol            5946 non-null   object
 11  Manual            5946 non-null   object
 12  5                 5946 non-null   object
 13  >5                5946 non-null   object
 14  make              5946 non-null   object
 15  model             5946 non-null   object
dtypes: object(16)
memory usage: 743.4+ KB


In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19820 entries, 0 to 19819
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   selling_price     19820 non-null  float64
 1   km_driven         19820 non-null  int64  
 2   mileage           19820 non-null  float64
 3   engine            19820 non-null  float64
 4   max_power         19820 non-null  float64
 5   age               19820 non-null  float64
 6   make              19820 non-null  object 
 7   model             19820 non-null  object 
 8   Individual        19820 non-null  int64  
 9   Trustmark Dealer  19820 non-null  int64  
 10  Diesel            19820 non-null  int64  
 11  Electric          19820 non-null  int64  
 12  LPG               19820 non-null  int64  
 13  Petrol            19820 non-null  int64  
 14  Manual            19820 non-null  int64  
 15  5                 19820 non-null  int64  
 16  >5                19820 non-null  int64 

In [28]:
# checking if any column is missing in training set except 'selling_price'

for col in df.columns:
  c = col not in X_train_simple
  print(f'{c} : {col}')


True : selling_price
False : km_driven
False : mileage
False : engine
False : max_power
False : age
False : make
False : model
False : Individual
False : Trustmark Dealer
False : Diesel
False : Electric
False : LPG
False : Petrol
False : Manual
False : 5
False : >5



### ⚠ Important Business Note
Imputation is NOT just statistical.
Wrong imputation can:
- Introduce bias
- Break fairness
- Mislead ML models

Always validate with business context.


In real systems, bad data is not always missing — sometimes it is present but abnormal.

Examples from a car-pricing / loan-risk system:

- selling_price = 5 ❌ (too low)
- selling_price = 10,00,00,000 ❌ (too high)
- km_driven = 2,000,000 ❌
- Sudden spike in average prices ❌

Anomaly rules detect values that are technically valid but business-impossible or suspicious.

### Types of Anomaly Rules

| Layer           | Purpose                 |
| --------------- | ----------------------- |
| **Rule-based**  | Business limits         |
| **Statistical** | Distribution violations |
| **Model-based** | Pattern deviations      |


## 5. Anomaly Detection Rules

An anomaly is a value that deviates significantly from expected behavior.

### Rule-Based Anomaly Detection
- Threshold rules
- Percentile-based rules
- Z-score rules


In [3]:
# Import and load data
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest

df = pd.read_csv('dataset/cars24-car-price-cleaned-new.csv')
df.describe()

,selling_price,km_driven,mileage,engine,max_power,age,Individual,Trustmark Dealer,Diesel,Electric,LPG,Petrol,Manual,5,>5
count,19820.000000,1.982000e+04,19820.000000,19820.000000,19820.000000,19820.000000,19820.000000,19820.000000,19820.000000,19820.000000,19820.000000,19820.000000,19820.000000,19820.000000,19820.000000
mean,6.585509,5.815856e+04,19.503402,1475.702381,98.122907,8.438547,0.390666,0.009586,0.492583,0.000404,0.003229,0.487841,0.802674,0.835015,0.152825
std,4.847364,5.171563e+04,4.297784,518.571223,44.761727,3.196636,0.487912,0.097442,0.499958,0.020087,0.056734,0.499865,0.397990,0.371176,0.359828
min,0.300000,1.000000e+02,4.000000,0.000000,5.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,3.410000,3.100000e+04,16.950000,1197.000000,73.900000,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000
50%,5.200000,5.200000e+04,19.300000,1248.000000,86.800000,8.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000
75%,7.850000,7.400000e+04,22.320000,1582.000000,112.000000,10.000000,1.000000,0.000000,1.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000
max,20.902500,3.800000e+06,120.000000,6752.000000,626.000000,31.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [4]:
# train-test split

train_df, test_df = train_test_split(
    df, test_size = 0.3, random_state = 42
)

### ANOMALY RULE 1: Rule-Based (Business Constraints)

Business teams know absolute limits better than algorithms.

Example Rules

- Price must be between ₹50,000 and ₹50,00,000
- Km driven must be ≤ 500,000
- Manufacturing year must be realistic


In [5]:
rule_anomalies = test_df[
    (test_df['selling_price'] < 50000) |
    (test_df['selling_price'] > 5000000) |
    (test_df['km_driven'] > 500000)
]

print(f'Rule based anomalies: {rule_anomalies.shape[0]}')

Rule based anomalies: 5946


### ANOMALY RULE 2: Statistical (Z-Score / IQR)

In [6]:
Q1 = train_df['selling_price'].quantile(0.25)
Q3 = train_df['selling_price'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

stat_anomalies = test_df[
    (test_df['selling_price'] < lower_bound) |
    (test_df['selling_price'] > upper_bound)
]

print(f'Statistical anomalies: {stat_anomalies.shape[0]}')

Statistical anomalies: 545


### ANOMALY RULE 3: Model-Based (Isolation Forest)
Why this rule?

    - Learns normal behavior
    - Flags multi-feature anomalies
    - Industry-standard for unsupervised anomaly detection

In [7]:
num_cols = ['selling_price', 'km_driven']

In [8]:
# Train isolation forest (ONLY on training data)
iso_forest = IsolationForest(
    contamination = 0.02,  # 2% expected anomalies
    random_state = 42
)

iso_forest.fit(train_df[num_cols])

IsolationForest(contamination=0.02, random_state=42)

In [10]:
test_df['anomaly_flag'] = iso_forest.predict(test_df[num_cols])

#-1 → anomaly ,1 → normal
model_anomalies = test_df[test_df['anomaly_flag'] == -1]
print(f'Model-based anomalies: {model_anomalies.shape[0]}')

Model-based anomalies: 136


Combining All Anomaly Rules (Production Pattern)

In [11]:
all_anomalies = pd.concat([
    rule_anomalies,
    stat_anomalies,
    model_anomalies
]).drop_duplicates()

print(f'Total unique anomalies detected: {all_anomalies.shape[0]}')

Total unique anomalies detected: 6074



## 6. When to FAIL vs WARN a Pipeline

### FAIL when:
- Compliance is violated
- Business-critical metrics are wrong
- Model inputs are corrupted

### WARN when:
- Minor drift detected
- Non-critical fields missing

This decision is **business-driven**, not technical.



## 7. End-to-End Data Quality Pipeline (Conceptual Flow)

1. Ingest raw data
2. Validate schema & expectations
3. Handle missing values
4. Detect anomalies
5. Generate reports & alerts
6. Allow or block downstream usage

This pipeline ensures **trustworthy data**.



## 8. Key Takeaways

- Data Quality is infrastructure, not cleanup
- Great Expectations enforces data contracts
- Imputation must align with business logic
- Anomaly rules prevent silent failures
- Production pipelines must be auditable

You now have the foundation to build **enterprise-grade data quality systems**.
